# Clear sky example (RFMIP)


## Overview

This notebook demonstrates the use of pyRTE-RRTMP to solve the simple problem of computing 
   clear-sky broadband (spectrally-integrated) fluxes. The examples use a set of atmospheric 
   conditions used in the Radiative Forcing Model Intercomparison Project. The conditions 
   are described in [this paper](https://doi.org/10.1029/2020JD033483). The conditions, as 
   well as the results for the reference Fortran implementation of RTE-RRTMGP, are downloaded 
   by the Python package. 
   
 Although they are part of the same Python package we distinguish between `pyRRTMGP`, 
   which converts a description of the atmosphere into a radiative transfer problem, and 
   `pyRTE` which solves the radiative transfer problem to determine broadband fluxes

pyRTE-RRTMGP relies on `xarray` representions of data and `dask` for parallelization 

## The workflow 

For both longwave and shortwave problems we will 
1. Initialize pyRRTMGP by reading the gas optics data
2. Read the RFMIP atmospheric conditions
3. Compute spectrally-resolved gas optics properties 
4. Solve the radiative transfer equation to obtain upward and downward fluxes
5. Check the results against the reference solutions generated with the original RTE fortran code


# Setting up the problem 

## Dependencies

In [1]:
import numpy as np
import xarray as xr

from pyrte_rrtmgp.rrtmgp.data_files import (
    CloudOpticsFiles,
    GasOpticsFiles,
)
from pyrte_rrtmgp import rte
from pyrte_rrtmgp.rrtmgp import GasOptics
from pyrte_rrtmgp.examples import RFMIP_FILES, load_example_file

## Initialize pyRRTMGP gas optics calculations 

In [2]:
gas_optics_lw = GasOptics(
    gas_optics_file=GasOpticsFiles.LW_G256
)

gas_optics_sw = GasOptics(
    gas_optics_file=GasOpticsFiles.SW_G224
)

## Read the RFMIP atmopheric profiles

In [3]:
atmosphere = load_example_file(RFMIP_FILES.ATMOSPHERE)

In [4]:
atmosphere

<xarray.Dataset> Size: 2MB
Dimensions:                  (site: 100, expt: 18, layer: 60, level: 61)
Dimensions without coordinates: site, expt, layer, level
Data variables: (12/66)
    lon                      (site) float32 400B 27.0 24.0 162.0 ... 250.5 246.0
    lat                      (site) float32 400B -28.5 28.5 31.5 ... -3.0 -24.0
    time                     (site) float32 400B 4.25 14.5 14.5 ... 363.0 363.0
    sst                      (site) float32 400B -9.99 -9.99 ... 298.1 297.0
    expt_label               (expt) <U49 4kB 'Present day (PD)' ... 'LGM'
    pres_layer               (site, layer) float32 24kB 10.0 29.21 ... 1.014e+05
    ...                       ...
    hfc134a_GM               (expt) float32 72B 80.52 0.0 80.52 ... 421.4 0.0
    c6f14_GM                 (expt) float32 72B 0.279 0.0 0.279 ... 0.5222 0.0
    hcfc141b_GM              (expt) float32 72B 23.81 0.0 23.81 ... 1.285 0.0
    chcl3_GM                 (expt) float32 72B 9.902 6.0 9.902 ... 5.876 0.0
    c2f6_GM                  (expt) float32 72B 4.399 1.289e-06 ... 7.174 0.0
    cfc11eq_GM               (expt) float32 72B 809.2 32.11 ... 1.939e+03 0.0
Attributes: (12/25)
    title:               Atmospheric conditions for off-line radiative transf...
    institution_id:      UColorado
    institution:         University of Colorado, Boulder, CO 80309, USA
    activity_id:         input4MIPs
    Conventions:         CF-1.6
    creation_date:       2019-03-20 16:07:21-0400
    ...                  ...
    nominal_resolution:  10 km
    target_mip:          RFMIP
    variable_id:         multiple
    grid_label:          none
    tracking_id:         hdl:21.14100/f379c294-f7bb-442d-bca7-64661d60780e
    license:             Atmospheric condition data for RFMIP produced by the...

Layer pressures and temperatures are bounded by range of the empirical 
  data. Level pressures are only restricted to be > 0 but the reference 
  results were produced using the minimum allowed layer pressure. 
  We reproduce that restriction here to get the same answers 
  as the reference calculation. 

In [6]:
atmosphere["pres_level"] = xr.ufuncs.maximum(
    gas_optics_sw.press_min,
    atmosphere["pres_level"],
)

## Conform to expectations

pyRRTMGP interprets the input `xr.Dataset` by looking for `xr.DataArray`s with specific names. 
  Gases are specified with their chemical formula (or, for halocarbons, by an abbreviation), and 
  volume mixing ratios are expected in absolute units, so the RFMIP datasets needs some manipulation. 


In [7]:
gas_names = {
    "water_vapor": "h2o",
    "carbon_dioxide_GM": "co2",
    "ozone":"o3",
    "nitrous_oxide_GM": "n2o",
    "carbon_monoxide_GM": "co",
    "methane_GM": "ch4",
    "oxygen_GM": "o2",
    "nitrogen_GM": "n2",
    "carbon_tetrachloride_GM": "ccl4",
    "cfc11_GM": "cfc11",
    "cfc12_GM": "cfc12",
    "hcfc22_GM": "cfc22",
    "hfc143a_GM": "hfc143a",
    "hfc125_GM": "hfc125",
    "hfc23_GM": "hfc23",
    "hfc32_GM": "hfc32",
    "hfc134a_GM": "hfc134a",
    "cf4_GM": "cf4",
}

atmosphere = atmosphere.rename_vars(gas_names)
for g in gas_names.values(): 
    if hasattr(atmosphere[g], "units"):
        atmosphere[g] *= float(atmosphere[g].units)
        atmosphere[g].assign_attrs({"units":"1"})

In [9]:
atmosphere

<xarray.Dataset> Size: 2MB
Dimensions:                 (site: 100, expt: 18, layer: 60, level: 61)
Dimensions without coordinates: site, expt, layer, level
Data variables: (12/66)
    lon                     (site) float32 400B 27.0 24.0 162.0 ... 250.5 246.0
    lat                     (site) float32 400B -28.5 28.5 31.5 ... -3.0 -24.0
    time                    (site) float32 400B 4.25 14.5 14.5 ... 363.0 363.0
    sst                     (site) float32 400B -9.99 -9.99 ... 298.1 297.0
    expt_label              (expt) <U49 4kB 'Present day (PD)' ... 'LGM'
    pres_layer              (site, layer) float32 24kB 10.0 29.21 ... 1.014e+05
    ...                      ...
    hfc134a                 (expt) float32 72B 8.052e-11 0.0 ... 4.214e-10 0.0
    c6f14_GM                (expt) float32 72B 0.279 0.0 0.279 ... 0.5222 0.0
    hcfc141b_GM             (expt) float32 72B 23.81 0.0 23.81 ... 0.0 1.285 0.0
    chcl3_GM                (expt) float32 72B 9.902 6.0 9.902 ... 6.0 5.876 0.0
    c2f6_GM                 (expt) float32 72B 4.399 1.289e-06 ... 7.174 0.0
    cfc11eq_GM              (expt) float32 72B 809.2 32.11 ... 1.939e+03 0.0
Attributes: (12/25)
    title:               Atmospheric conditions for off-line radiative transf...
    institution_id:      UColorado
    institution:         University of Colorado, Boulder, CO 80309, USA
    activity_id:         input4MIPs
    Conventions:         CF-1.6
    creation_date:       2019-03-20 16:07:21-0400
    ...                  ...
    nominal_resolution:  10 km
    target_mip:          RFMIP
    variable_id:         multiple
    grid_label:          none
    tracking_id:         hdl:21.14100/f379c294-f7bb-442d-bca7-64661d60780e
    license:             Atmospheric condition data for RFMIP produced by the...

# Compute the spectrally-dependent optical properties 

For the longwave problem we will make a new dataset with the optical properties 
  pyRRTMGP compute the optical properties (just optical depth `tau` for the longwave problem) and three 
  radiation source functions (on layers, on levels, and at the surface)

In [8]:
optical_props = gas_optics_lw.compute(
    atmosphere,
    add_to_input = False,
)
optical_props

<xarray.Dataset> Size: 675MB
Dimensions:                  (site: 100, expt: 18, gpt: 256, layer: 60,
                              level: 61, bnd: 16, pair: 2)
Coordinates:
  * site                     (site) int64 800B 0 1 2 3 4 5 ... 94 95 96 97 98 99
  * expt                     (expt) int64 144B 0 1 2 3 4 5 ... 12 13 14 15 16 17
  * gpt                      (gpt) int64 2kB 0 1 2 3 4 5 ... 251 252 253 254 255
Dimensions without coordinates: layer, level, bnd, pair
Data variables:
    surface_source           (gpt, site, expt) float64 4MB 0.6704 ... 2.574e-06
    layer_source             (layer, gpt, site, expt) float64 221MB 0.6256 .....
    level_source             (level, gpt, site, expt) float64 225MB 0.6256 .....
    surface_source_jacobian  (gpt, site, expt) float64 4MB 0.003301 ... 1.229...
    tau                      (layer, gpt, site, expt) float64 221MB 1.694e-08...
    bnd_limits_gpt           (bnd, pair) int32 128B 1 16 17 32 ... 240 241 256
Attributes:
    top_at_1:  <xarray.DataArray 'pres_layer' ()> Size: 1B\narray(True)\nAttr...

For the shortave problem we will append the optical properties to the original dataset 
   Shortwave problems have three optical properties (`tau`, `ssa`, and `g`) but a single 
   source function defined at the top of atmosphere 

In [ ]:
gas_optics_sw.compute(
    atmosphere,
)
atmosphere

## Solve the Radiative Transfer Equation

Before we can solve the radiative transfer equation we need to specify the boundary conditions - 
  for longwave radiation, the `surface_emissivity` which here comes from the RFMIP conditions 

In [ ]:
optical_props["surface_emissivity"] = atmosphere.surface_emissivity

With the problem specified (optical properties, source functions, boundary conditions), 
  we can now solve the radiative transfer equation to find
  the upward and downward broadband radiative fluxes for each atmospheric profile.

For the longwave problem we use the dataset containing only the radiative transfer problem. 
All the arrays for the shortwave are in the same dataset

In [ ]:
lw_fluxes = optical_props.rte.solve(
	add_to_input=False, 
)

atmosphere.rte.solve() 

## Check the results against the reference solutions 

We compare all fluxes (up and down, shortwave and longwave) against the results of the reference code to ensure we 
   have the same results to within some tolerance 

### Read the reference results 

In [ ]:
ref = xr.merge([
	load_example_file(RFMIP_FILES.REFERENCE_RLU), 
	load_example_file(RFMIP_FILES.REFERENCE_RLD),
	load_example_file(RFMIP_FILES.REFERENCE_RSU), 
	load_example_file(RFMIP_FILES.REFERENCE_RSD),
	], 
    compat='equals', 
)

### Compare longave results

In [ ]:
assert np.isclose(
    lw_fluxes["lw_flux_up"].transpose("expt", "site", "level"),
    ref["rlu"],
    atol=1e-7,
).all(), "Longwave flux up mismatch"
assert np.isclose(
    lw_fluxes["lw_flux_down"].transpose("expt", "site", "level"),
    ref["rld"],
    atol=1e-7,
).all(), "Longwave flux down mismatch"

### Compare shortwave results

In [ ]:
assert np.isclose(
    atmosphere["sw_flux_up"].transpose("expt", "site", "level"),
    ref["rsu"],
    atol=1e-7,
).all(), "Shortwave flux up mismatch"
assert np.isclose(
    atmosphere["sw_flux_down"].transpose("expt", "site", "level"),
    ref["rsd"],
    atol=1e-7,
).all(), "Shortwave flux down mismatch"

In [ ]:
print("RFMIP clear-sky calculations validated")

# Variants

See the `pyRTE-quick-start` notebook for more examples, including how to parallelize computations with `dask`,  
  how to add clouds to the problem, and how to combine multiple steps of the calculation at once.